In [4]:
import hashlib
import multiprocessing.managers
from matplotlib import pyplot as plt
from operator import mul
from hypercube import *
import itertools, math
import tqdm
import multiprocessing, ctypes, contextlib
import time, csv, json, enum

from pyhypercube_local_run import createFilesForSubmit


def run(job: HypercubeJob):
    res = job.run()

def runSingleThreaded(job: HypercubeJob):
    res = job.runSingleThreaded()

In [5]:
maxMemoryMB = 32*1024**2
maxPoints = int(1e+8)
jobs = createFilesForSubmit(maxMemoryMB, maxPoints)
mem = max([j.maxMemory() for j in jobs])
print(f'Jobs count {len(jobs)}, maxMem = {mem/1024**2} MB')

Jobs count 15427, maxMem = 76.2939453125 MB


In [6]:
times = {}
nprocesses = range(12, 72, 4)
for processes in nprocesses:
    with multiprocessing.Pool(processes) as pool:
        print(f'Benchmarking processes = {processes}')
        t = -time.time()        
        for it in tqdm.tqdm(pool.imap(run, jobs), total = len(jobs), leave=False):
            pass
        t += time.time()
        times[processes] = t
        print(f'Benchmarking processes = {processes} finished, time = {t} secs')

timesSingleThreaded = {}
for processes in nprocesses:
    with multiprocessing.Pool(processes) as pool:
        print(f'Benchmarking processes = {processes}')
        t = -time.time()        
        for it in tqdm.tqdm(pool.imap(runSingleThreaded, jobs), total = len(jobs), leave=False):
            pass
        t += time.time()
        timesSingleThreaded[processes] = t
        print(f'Benchmarking processes = {processes} finished, time = {t} secs')


print(f"Elapsed {time.time() - t} secs")

fig = plt.figure()
plt.plot(times.keys(), times.values())
plt.plot(timesSingleThreaded.keys(), timesSingleThreaded.values())
plt.xlabel('number of processes')
plt.ylabel('total time')
plt.yscale('log')
plt.legend(['run', 'runSingleThreaded'])

Benchmarking processes = 12


  0%|          | 0/15427 [00:00<?, ?it/s]

KeyboardInterrupt: 